In [1]:
import rasterio
from pyproj import CRS, Transformer
import numpy as np
import os
#import matplotlib.pyplot as plt
import cartopy.crs as ccrs
#from matplotlib import ticker
import plotly.graph_objects as go

# Bathymetry file in geotif format
file_path = r'C:\Users\SFr\Work\VolumeSampler\Bathymetri\bathy_projected.tif'

In [2]:
with rasterio.open(file_path) as src:
    # Read the elevation or data values (assume a single band)
    data = src.read(1)

    # Get the CRS (EPSG:3065) and transform to lat/lon (EPSG:4326)
    original_crs = src.crs
    transform_affine = src.transform
    
    # Get the x and y coordinates in the original projection (EPSG:3065)
    width = src.width
    height = src.height
    cols, rows = np.meshgrid(np.arange(width), np.arange(height))
    xs, ys = rasterio.transform.xy(transform_affine, rows, cols)
    xs = np.array(xs)
    ys = np.array(ys)

In [3]:
# Step 2: Reproject coordinates from EPSG:3065 to EPSG:4326
#projected_crs = CRS.from_epsg(3065)
#target_crs = CRS.from_epsg(4326)

transformer = Transformer.from_crs("EPSG:3065", "EPSG:4326", always_xy=True)
# Transform from projected CRS to lat/lon using rasterio.warp.transform
lon, lat = transformer.transform(xs.flatten(), ys.flatten())
lon = np.array(lon).reshape(data.shape)
lat = np.array(lat).reshape(data.shape)

In [4]:
# Simulations results directory
Res_dir = r'L:\VolumeSampler\slopeunits\Results'
# Directory to store the result figures
Res_dir_out = r'C:\Users\SFr\Work\VolumeSampler\slopeunits\Results'
# List all simulations
Res_dirs = os.listdir(Res_dir)

In [6]:
# Loop through the simulations
for rd in Res_dirs:
    # Read result data
    dd = rasterio.open(os.path.join(Res_dir,rd,'slumap_clean.tif'))
    # Read channel 1, this is assumed to be elevations
    dataC = dd.read(1)
    # Remove nan data and set to 0
    dataC[dataC > 60000] = 0

    # Split into 50 categories that each represent a color
    # This can be adjusted for best visibility
    numC = 50
    a = np.arange(1,numC+1)
    c = 0;
    # Loop through the individual catogiries and limit between 0 and 50
    for i in range(np.max(dataC)):
        dataC[dataC == i+1] = a[c]
        c += 1
        if c > numC-1:
            c = 0

    # Plot the figure
    # Step 2: Create the 3D plot with Plotly
    fig = go.Figure(data=[go.Surface(z=data, x=lon, y=lat, surfacecolor = dataC, colorscale='Viridis')])
    
    # Customize layout
    fig.update_layout(
        title='3D GeoTIFF Elevation Plot',
        scene=dict(
            xaxis_title='Longitude',
            yaxis_title='Latitude',
            zaxis_title='data'
        )
    )

    # make result directory if it does not exist
    os.makedirs(os.path.join(Res_dir_out,rd), exist_ok=True)
    
    # Step 3: Save the plot as an interactive HTML file
    output_file = os.path.join(Res_dir_out,rd,'interactive_3d_geotiff_plot.html')
    fig.write_html(output_file)
    
    print(f'Interactive 3D plot saved as {output_file}')

Interactive 3D plot saved as C:\Users\SFr\Work\VolumeSampler\slopeunits\Results\t1000000_a1000000_c0.2_rf20\interactive_3d_geotiff_plot.html
Interactive 3D plot saved as C:\Users\SFr\Work\VolumeSampler\slopeunits\Results\t1000000_a1000000_c0.2_rf40\interactive_3d_geotiff_plot.html
Interactive 3D plot saved as C:\Users\SFr\Work\VolumeSampler\slopeunits\Results\t1000000_a1000000_c0.2_rf60\interactive_3d_geotiff_plot.html
Interactive 3D plot saved as C:\Users\SFr\Work\VolumeSampler\slopeunits\Results\t1000000_a1000000_c0.4_rf20\interactive_3d_geotiff_plot.html
Interactive 3D plot saved as C:\Users\SFr\Work\VolumeSampler\slopeunits\Results\t1000000_a1000000_c0.4_rf40\interactive_3d_geotiff_plot.html
Interactive 3D plot saved as C:\Users\SFr\Work\VolumeSampler\slopeunits\Results\t1000000_a1000000_c0.4_rf60\interactive_3d_geotiff_plot.html
Interactive 3D plot saved as C:\Users\SFr\Work\VolumeSampler\slopeunits\Results\t1000000_a1000000_c0.6_rf20\interactive_3d_geotiff_plot.html
Interactive 3

RasterioIOError: L:\VolumeSampler\slopeunits\Results\t500000_a500000_c0.1_rf70\slumap_clean.tif: No such file or directory